In [1]:
from collections import Counter, defaultdict, deque, namedtuple
from copy import deepcopy
import functools
import inspect
import json
import os
from pathlib import Path
import pickle
from pprint import pp, pprint
import re
import sys
import time
from typing import Dict, List

import numpy as np
import pandas as pd
import plotly.express as px

from colorutils import Color

from dotenv import load_dotenv
from tqdm import tqdm

from aic_nlp_utils.json import read_jsonl, read_json, write_json, write_jsonl, process_to_jsonl
from aic_nlp_utils.pycfg import parse_pycfg_args, read_pycfg
%load_ext autoreload
%autoreload 2

from long_sum.utils import *

sys.path.append("/home/drchajan/devel/python/FC/automated-fact-checking")

os.environ['VLLM_WORKER_MULTIPROC_METHOD']='spawn'
load_dotenv()

/home/drchajan/devel/python/FC/VENV/vllm/lib/python3.11/site-packages/aic_nlp_utils/json.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


True

In [1]:
from object_aligner import ObjectAligner

In [4]:
# from long_sum.object_aligner import ObjectAligner


gold = "ahojky"
pred = "ahoj"

schema = {
    "type": "string"
}

# gold = 49
# pred = 50
# schema = {
#     "type": "integer",
# }

aligner = ObjectAligner("json_metric", schema)

# align_lists_ignore_order(gold, pred)
# print()
# print(aligner.align(gold, pred))
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 89%, let us align the predicted output to the gold and analyze the differences:
The predicted value "ahoj" does not match the gold "ahojky" (score=89%).


In [3]:

# gold = [5, 3, 4, 1, -1]
# pred = [5, 3, 4, 1]
# pred = [1, 5, 3, 1]
# pred = [-1, 1, 4, 3, 5]

gold = [1, 2, 4]
pred = [2, 3]

# gold = [4, 5]
# pred = [1, 2, 3]

schema = {
  "type": "array",
  "items": {
    "type": "integer",
    "score": "exact"
    # "score": "invdiff"
  }
}

# gold = []
# gold = ["car", "bus"]
# pred = ["cat", "bus"]
# pred = ["cat"]
# pred = []

gold = ["weight", "name", "age"]
pred = ["name", "ages", "title"]

# gold = ["cat", "dog", "hamster", "fly"]
# gold = ["fly", "cat", "hamster", "planet"]
# pred = ["hamster", "cat", "fly"]
# pred = ["cat", "dog", "elephant", "hamster", "cat"]
# pred = ["car", "haster", "dag", "cat"]
# pred = ["car", "hamter", "dog", "cat"]

schema = {
  "type": "array",
  "items": {
    "type": "string",
    # "score": "exact",
    "score": "jaro",
    "threshold": 0.5, # all < is set to zero
  },
  # "order": "fixed",
  "order": "align",
  # "ignore_excess": True,
  # "ignore_missing": True,
}

aligner = ObjectAligner("json_metric", schema)

# align_lists_ignore_order(gold, pred)
# print()
# pp(aligner.align(gold, pred))
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 48%, let us align the predicted output to the gold and analyze the differences:
The predicted list scores 48%:
  The predicted list item "title" is excessive, it was not in the gold.
  The predicted output misses the "weight" list item from the gold.
  The predicted value "name" exactly matches the gold.
  The predicted value "ages" does not match the gold "age" (score=92%).


In [100]:
gold = [["car", 5, "airplane"], ["bus", 3, "ship"]]
pred = [["cat", 5, "plane"], ["bus", 3]]

schema = {
  "type": "array",
  "items": {
    "type": "array",
    "prefixItems": [
        {"type": "string"},
        {"type": "integer"},
    ],
    "prefixWeights": [1, 1],
    
    "items": {"type": "string"}, 
       
    "prefixImportance": 2.0,
    "restImportance": 1.0,
  },
  "ignoreExcess": True,
  "ignoreMissing": True
}

aligner = ObjectAligner("json_metric", schema)
# ret = aligner.align(gold, pred)
# pp(ret)
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scored overall 76%, let us align the predicted output to the gold and analyze the differences:
MatchList score = 0.7643518518518517
  MatchList score = 0.8620370370370369
    The predicted value "cat" does not match the gold "car" (score=78%).
    The predicted value "5" exactly matches the gold.
    The predicted value "plane" does not match the gold "airplane" (score=81%).
  MatchList score = 0.6666666666666666
    The predicted value "bus" exactly matches the gold.
    The predicted value "3" exactly matches the gold.
    The predicted output misses the "ship" list item from the gold.



In [102]:
gold = [[1, 2, 3],    [4, 5], [6], [7, 8]]
pred = [[1, 2, 4, 5], [4, 5], [3], [7, 8]]
# pred = [[1, 2, 4, 5],  [7, 8], [4, 5, 2], [3]]
schema = {
  "type": "array",
  "items": {
    "type": "array",
    "items": {
      "type": "integer",
      # "score": "exact"
      "score": "invdiff"
    }
  },
  "order": "fixed",
  # "order": "align"
}

aligner = ObjectAligner("json_metric", schema)
# ret = aligner.align(gold, pred)
# pp(ret)
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 72%, let us align the predicted output to the gold and analyze the differences:
The predicted list scores 72%:
  The predicted list scores 62%:
    The predicted value "1" exactly matches the gold.
    The predicted value "2" exactly matches the gold.
    The predicted value "4" does not match the gold "3" (score=50%).
    The predicted list item "5" is excessive, it was not in the gold.
  The predicted list perfectly matches the gold one:
    The predicted value "4" exactly matches the gold.
    The predicted value "5" exactly matches the gold.
  The predicted list scores 25%:
    The predicted value "3" does not match the gold "6" (score=25%).
  The predicted list perfectly matches the gold one:
    The predicted value "7" exactly matches the gold.
    The predicted value "8" exactly matches the gold.



In [103]:
gold = [[[1, 2], [3]], [[4], [5, 6, 7]]]
pred = [[[4], [3]], [[4], [5, 6, 7]]]
# pred = [[[4], [3]], [[4], [5, 5, 8]]]
schema = {
    "type": "array",
    "items": {
        "type": "array",
        "items": {
            "type": "array",
            "items": {
                "type": "integer",
                # "score": "exact"
                "score": "invdiff"
                }
            }
        },
    # "order": "fixed",
  "order": "align"
}

aligner = ObjectAligner("json_metric", schema)
# pp(aligner.align(gold, pred))
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 79%, let us align the predicted output to the gold and analyze the differences:
The predicted list scores 79%:
  The predicted list scores 58%:
    The predicted list scores 17%:
      The predicted output misses the "1" list item from the gold.
      The predicted value "4" does not match the gold "2" (score=33%).
    The predicted list perfectly matches the gold one:
      The predicted value "3" exactly matches the gold.
  The predicted list perfectly matches the gold one:
    The predicted list perfectly matches the gold one:
      The predicted value "4" exactly matches the gold.
    The predicted list perfectly matches the gold one:
      The predicted value "5" exactly matches the gold.
      The predicted value "6" exactly matches the gold.
      The predicted value "7" exactly matches the gold.



In [117]:
gold = {"weight": 90, "name": "John", "age": 24}
# gold = {"name": "John", "age": 24}
pred = {"name": "Johny", "ages": 23, "title": "Mr."}

schema = {
  "type": "object",
  "properties": {
    "weight": {
      "type": "integer",
      "valueWeight": 1.0,
    },
    "name": {
      "type": "string",
      "score": "jaro",
      "valueWeight": 1.0,
    },
    "age": {
      "type": "integer",
      "valueWeight": 1.0,
    }
  },
  # "additionalProperties": False,
  # "keyScore": "exact",
  "keyScore": "jaro",
  "keyThreshold": 0.5,
  "keyImportance": 1.0,
  "valueImportance": 1.0,
}

aligner = ObjectAligner("json_metric", schema)
# pp(aligner.align(gold, pred))
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 42%, let us align the predicted output to the gold and analyze the differences:
  KEY = The predicted key "ages" does not match the gold "age" (score=92%).
  VALUE = The predicted value "23" does not match the gold "24" (score=50%).




In [116]:
gold = {"a": [1, 2], "b": [3, 4]}
pred = {"a": [1, 2], "b": [3, 5]}
# pred = {"aab": [1, 2], "b": [3, 4]}

schema = {
  "type": "object",
  "properties": {
    "a": {
      "type": "array",
      "items": {"type": "integer"},
      "valueWeight": 1.0, # this is default value
    },
    "b": {
      "type": "array",
      "items": {"type": "integer"},
      "valueWeight": 1.0,
    },
  },
  # "additionalProperties": False,
  "keyScore": "exact",
  # "keyScore": "jaro",
  "keyImportance": 0.0,
  "valueImportance": 1.0,
}

aligner = ObjectAligner("json_metric", schema)
# pp(aligner.align(gold, pred))
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 88%, let us align the predicted output to the gold and analyze the differences:
  KEY = The predicted key "b" does not match the gold "b" (score=100%).
  VALUE = The predicted list scores 75%:
    The predicted value "3" exactly matches the gold.
    The predicted value "5" does not match the gold "4" (score=50%).




In [119]:
gold = [{"a": 1, "b": 2}, {"a": 3, "b": 4}]
pred = [{"a": 1, "b": 2}, {"a": 3, "bbaa": 4}]

schema = {
  "type": "array",
  "items": {
      "type": "object",
      "properties": {
        "a": {
            "type": "integer",
            "valueWeight": 1.0,
        },
        "b": {
            "type": "integer",
            "valueWeight": 1.0,
        },
        },
    #   "keyScore": "jaro",
      "keyScore": "exact",
  },
}

aligner = ObjectAligner("json_metric", schema)
# pp(aligner.align(gold, pred))
print(aligner.metric(gold, pred)["reasoning"])

The predicted output scores overall 67%, let us align the predicted output to the gold and analyze the differences:
The predicted list scores 67%:
    KEY = The predicted key "b" exactly matches the gold.
    VALUE = The predicted value "2" exactly matches the gold.

    KEY = The predicted key "None" does not match the gold "b" (score=0%).
    VALUE = The predicted value "None" does not match the gold "4" (score=0%).


In [5]:
from object_aligner import ObjectAligner

schema = {"type": ["string", "null"], "nullScore": 0.8}
aligner = ObjectAligner(schema)

aligner.metric(None, None)         # {'score': 1.0}
aligner.metric(None, "Smith")      # {'score': 0.8}
aligner.metric("Smith", None)      # {'score': 0.8}
aligner.metric("Smith", "Smyth")   # primitive comparator (jaro)


{'score': 0.8666666666666667}

In [8]:
schema = {
    "type": "object",
    "properties": {
        "diagnosis":   {"type": ["string", "null"], "nullScore": 0.0},
        "middle_name": {"type": ["string", "null"], "nullScore": 0.8},
    },
}
aligner = ObjectAligner(schema, generate_feedback=True)

gold = {"diagnosis": "flu", "middle_name": None}
pred = {"diagnosis": None,  "middle_name": None}

aligner.metric(gold, pred)
# {'score': 0.75}


{'score': 0.75,
 'feedback': "The prediction scored 0.75 (deficit 0.25). Top 1 of 1 fix locations:\n1. /diagnosis: null/value mismatch (expected 'flu', got None). Fixing this recovers +0.250.\nFocus on null-value errors — they account for 100% of the deficit shown."}

In [ ]:
aligner.feedback(gold, pred)

FeedbackResult(score=0.75, text="The prediction scored 0.75 (deficit 0.25). Top 1 of 1 fix locations:\n1. /diagnosis: null/value mismatch (expected 'flu', got None). Fixing this recovers +0.250.\nFocus on null-value errors — they account for 100% of the deficit shown.", entries=(FeedbackEntry(rank=1, op_kind='null_value_replace', op='replace', path='/diagnosis', score_delta=0.25, score_delta_pct=25.0, gold='flu', pred=None, text="1. /diagnosis: null/value mismatch (expected 'flu', got None). Fixing this recovers +0.250.", pair_id=''),), style='gepa', truncated=False, n_total_ops=1, error_breakdown={})

FeedbackResult(score=0.75, text="The prediction scored 0.75 (deficit 0.25). Top 1 of 1 fix locations:\n1. /diagnosis: null/value mismatch (expected 'flu', got None). Fixing this recovers +0.250.\nFocus on null-value errors — they account for 100% of the deficit shown.", entries=(FeedbackEntry(rank=1, op_kind='null_value_replace', op='replace', path='/diagnosis', score_delta=0.25, score_delta_pct=25.0, gold='flu', pred=None, text="1. /diagnosis: null/value mismatch (expected 'flu', got None). Fixing this recovers +0.250.", pair_id=''),), style='gepa', truncated=False, n_total_ops=1, error_breakdown={})